# Poisson interaction range와 coupling에 대한 KNEEP 반응

기본 **production_v3**는 \(g=0.5,1,2\), \(\ell=1,2,4,8\)의 **12개 조합 + 단일 \(g=0\) 대조군**을 비교한다. 격자 32², \(D=0.1,\bar\phi=0\)이며 Euler–Maruyama와 매 solver step의 midpoint medium EP를 사용한다.

조건마다 train / validation / test **1024 / 64 / 128 trajectory**, 각 production **1024 step**으로 train pair **1,048,576개**를 만든다. radius=15, logical batch=1024, microbatch=128, LR=1e-3, weight decay=1e-4, iterations=10,000이다. **전체 validation loss가 정체될 때만 ReduceLROnPlateau로 LR을 낮춘다.**

초기 field는 \(\phi=\pm\sqrt{-a/b}=\pm1\)의 domain slab이며 \(x=L/4,3L/4\)에 주기적 interface가 있다. 이것은 **burn-in 전 초기조건**이며 burn 16,000 step 동안 모양이 바뀔 수 있다. production에서 wall이 남는다고 가정하지 않는다.

대규모 생성·학습은 아직 이 workspace에서 실행하지 않았다. 서버 CLI/SLURM은 병렬 실행을 제공하며 notebook은 저장된 평가를 CPU에서 다시 표시할 수 있다.


## 비교하는 양

\[
d\phi=M\nabla^2(\mu_{\rm local,C}+gK_\ell*\phi+\lambda|\nabla\phi|^2)dt+\text{conserved noise},
\qquad \widehat K_\ell(q)=e^{-\ell q},\quad\widehat K_\ell(0)=0.
\]

각 조건은 실제 동역학을 새로 적분한다. 같은 trajectory 번호의 초기조건·noise seed는 조건 사이에서 공유하고 train / validation / test는 분리한다. 고정 g에서도 ell 증가 시 모든 nonzero Fourier mode가 약해지므로 EP/fluctuation의 단조 변화를 미리 가정하지 않는다.

물리 reference는 **system entropy를 제외한 medium EP**다. KNEEP는 observed pair의 비가역성을 추정하므로 finite-time map과 shell fluctuation이 medium reference와 같아야 하는 것은 아니다. 물리 shell과 학습 branch는 최소영상 Chebyshev 거리를 쓰지만 분해의 gauge가 다르다.


In [ ]:
%matplotlib inline
from pathlib import Path
from dataclasses import asdict
import os
import sys

def find_kneep_root(start: Path) -> Path:
    for parent in (start, *start.parents):
        for candidate in (parent, parent / "KNEEP"):
            if (candidate / "shell_force.py").is_file() and (candidate / "models").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("KNEEP 또는 그 상위 workspace에서 notebook을 실행하세요.")

KNEEP_ROOT = find_kneep_root(Path.cwd())
LRAMB_ROOT = KNEEP_ROOT.parent / "LRAMB"
if str(KNEEP_ROOT) not in sys.path:
    sys.path.insert(0, str(KNEEP_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
from models.lramb_poisson import dataset_preset, dataset_conditions, generate_datasets, read_manifest
from scripts.lramb_poisson_range import fit_preset, train_sweep, evaluate_sweep, load_evaluation_records
from utils.poisson_range_plots import (
    condition_label, group_records_by_strength, plot_training_history,
    plot_kernel_diagnostics, plot_window_averaged_ep, plot_ep_performance,
    plot_time_averaged_ep, plot_ep_spectrum, plot_fluctuation_comparison,
)
from utils.poisson_range_movies import animate_test_trajectory
torch.set_num_threads(1)
plt.rcParams.update({"figure.dpi": 105, "axes.titlesize": 10, "axes.labelsize": 9})
print("KNEEP:", KNEEP_ROOT)
print("LRAMB:", LRAMB_ROOT)
print("CUDA available:", torch.cuda.is_available())


## 설정

| 항목 | production_v3 |
|---|---:|
| g / ell | (0.5, 1, 2) / (1, 2, 4, 8), 대조군 g=0 한 번 |
| D / 평균 / dt | 0.1 / 0 / 0.01 |
| 격자 / 초기조건 | 32² / 주기적 domain wall, bulk ±1 |
| burn / production steps | 16,000 / 1,024 |
| 독립 train / validation / test | 1024 / 64 / 128 |
| train pair 수 / 조건 | 1,048,576 |
| radius / width / depth | 15 / 32 / 2 |
| batch / microbatch / iterations | 1024 / 128 / 10,000 |
| LR / weight decay | 1e-3 / 1e-4 |
| full validation 간격 | 250 iterations |
| plateau factor / patience / cooldown | 0.5 / 3 validation events / 1 event |
| absolute threshold / min LR / warmup | 1e-3 / 1e-5 / 0 |
| 저장 movie / 평균 window | 첫 test trajectory / 16 observed intervals |

\(a=-1,b=\kappa=M=1,\lambda=0.8\)이다. 음수 validation loss이므로 absolute threshold를 쓴다. PyTorch patience=3은 **충분한 개선이 없는 validation이 4회 연속일 때** LR을 낮춘다. microbatch마다 scheduler를 실행하지 않는다.

production 시간은 trajectory당 10.24다. trajectory 수가 느린 모드의 시간 축 수렴을 대신하지는 않는다. radius는 n/2보다 작게 두며 n=32, radius=15는 반대편 radius=16 행·열을 제외한다.

서버 병렬 실행:

    cd /pds/pds1/slurm_workspace/idh1203/KNEEP
    sbatch scripts/lramb_poisson_range.slurm
    python scripts/lramb_poisson_range.py all --preset production_v3 --num-gpus 4
    python scripts/lramb_poisson_range.py plot --preset production_v3


In [ ]:
PROFILE = os.environ.get("KNEEP_POISSON_PROFILE", "production_v3")
DATA_DEVICE = os.environ.get("KNEEP_POISSON_DATA_DEVICE", "auto")
FIT_DEVICE = os.environ.get("KNEEP_POISSON_FIT_DEVICE", "auto")
special_outputs = {
    "production_v3": KNEEP_ROOT / "results" / "lramb_poisson_production_v3",
    "production_v2": KNEEP_ROOT / "results" / "lramb_poisson_production_v2_range15",
}
DEFAULT_OUTPUT = special_outputs.get(PROFILE, KNEEP_ROOT / "results" / "lramb_poisson_range" / PROFILE)
OUTPUT_DIR = Path(os.environ.get("KNEEP_POISSON_OUTPUT_DIR", str(DEFAULT_OUTPUT))).expanduser().resolve()
DATA_CONFIG = dataset_preset(PROFILE)
FIT_CONFIG = fit_preset(PROFILE)
FORCE_GENERATE = False
FORCE_TRAIN = False
# 새 학습 뒤 처음 평가할 때 True. 기존 결과를 그릴 때는 False로 둔다.
EVALUATE_NEW = False
FORCE_EVALUATE = False
SHARD_CACHE_SIZE = 128
WINDOW_STEPS = 64
TEST_TRAJECTORY = 0
LOG_SCALE = True
MOVIE_CONDITION = None  # 예: "g_1_ell_4"; None이면 movie가 있는 첫 상호작용 조건.
MOVIE_KIND = "combined"  # "profile", "medium", "predicted", "combined"
MOVIE_FPS = 10
MOVIE_MAX_FRAMES = 64

assert FIT_CONFIG.max_distance < DATA_CONFIG.n / 2
if PROFILE == "production_v3":
    assert DATA_CONFIG.train_pairs == 1_048_576
    assert len(dataset_conditions(DATA_CONFIG)) == 13
display(Markdown(f"**Profile:** {PROFILE} · **결과 폴더:** {OUTPUT_DIR}"))
display({"dataset": DATA_CONFIG.to_dict(), "fit": asdict(FIT_CONFIG)})
print("condition IDs:", [item["condition_id"] for item in dataset_conditions(DATA_CONFIG)])
print(f"관측 간격={DATA_CONFIG.interval:g}, production 시간/trajectory={DATA_CONFIG.duration:g}")


## 1. 데이터 생성

각 g,ell 및 단일 g=0 조건의 split을 별도 trajectory shard로 저장한다. v3 float32 field는 train 전체 약 55.9 GB, val/test 포함 약 66.4 GB이고 reference·diagnostic·movie window·checkpoint는 추가다. mmap도 OS page cache를 사용하므로 RAM 상한을 보장하지 않는다.

\[
\Delta S_{\rm med}=-\frac{dx^2}{D}\sum_x
(\mu_{\rm local,C}+\mu_K+\mu_A)_{\rm midpoint}\Delta\phi_x
\]
를 모든 solver step에서 계산한다. 첫 test trajectory에는 profile과 **매 solver step의 medium EP를 16 observed intervals씩 평균한 map**도 저장한다. endpoint field 차이로 영상을 재구성하지 않는다.

이 cell은 순차 실행이다. 큰 sweep에는 CLI/SLURM을 사용한다.


In [ ]:
dataset_manifest = generate_datasets(
    DATA_CONFIG, OUTPUT_DIR, lramb_root=LRAMB_ROOT,
    device=DATA_DEVICE, progress=True, force=FORCE_GENERATE,
)
display({
    "dataset_complete": read_manifest(OUTPUT_DIR)["complete"],
    "condition_count": len(dataset_conditions(DATA_CONFIG)),
    "dataset_file": str(OUTPUT_DIR / "dataset.json"),
})


## 2. 저장된 train / validation으로 학습

정규화는 train만, checkpoint 선택과 scheduler는 전체 validation만 사용한다. test는 읽지 않는다. 각 조건의 3개 학습 seed는 독립적으로 fit한다.

기존 additive shell 모델의 nonlocal branch는 해당 shell 이웃을 입력받는다. 일반적인 중심–이웃 교차항을 표현하지 못할 수 있으므로 데이터와 scheduler 변경이 exact recovery를 보장하지 않는다.


In [ ]:
training_manifest = train_sweep(
    OUTPUT_DIR, FIT_CONFIG, device=FIT_DEVICE,
    shard_cache_size=SHARD_CACHE_SIZE, progress=True, force=FORCE_TRAIN,
)
display({
    "training_complete": training_manifest["complete"],
    "repeats": FIT_CONFIG.repeats,
    "max_distance": FIT_CONFIG.max_distance,
    "checkpoint_directory": str(OUTPUT_DIR / "checkpoints"),
})


## 3. 평가 또는 저장된 평가 읽기

**새 학습 직후에는 설정 cell에서 EVALUATE_NEW=True로 바꾸고 설정·이 cell을 실행한다.** 기본 False는 저장된 평가만 읽는다. missing cache가 inference나 학습을 자동 실행하지 않는다. FORCE_EVALUATE=True는 같은 signature의 평가도 다시 계산한다.

서버 all/evaluate가 끝난 결과는 setup → 설정 → 이 cell → 그림 cell 순서로 실행하고 생성·학습은 건너뛴다. 이전 결과는 OUTPUT_DIR로 폴더를 지정하면 된다.

KNEEP score의 공간 평균 자체가 global increment다. 학습 map의 공간 합은 global mean rate이며 물리 density map 적분에는 \(dx^2\)를 곱한다. medium EP를 stationary total EP의 exact target으로 해석하지 않는다.


In [ ]:
if EVALUATE_NEW:
    records = evaluate_sweep(
        OUTPUT_DIR, device=FIT_DEVICE,
        batch_size=FIT_CONFIG.training.prediction_batch_size,
        progress=True, force=FORCE_EVALUATE,
    )
else:
    try:
        records = load_evaluation_records(OUTPUT_DIR)
    except FileNotFoundError as error:
        raise FileNotFoundError(
            "저장된 평가가 없습니다. 새 학습 뒤라면 EVALUATE_NEW=True로 바꾸거나 "
            "CLI evaluate 단계를 먼저 실행하세요. 생성·학습은 자동 실행하지 않습니다."
        ) from error

# predicted: [fit seed, independent test trajectory, interval, branch]
# reference: [independent test trajectory, interval, physical component]
for record in records:
    predicted_rate = record["predicted_increment"].sum(-1).mean(-1) / record["interval"]
    predicted_map_rate = record["predicted_map"].sum((-3, -2, -1))
    reference_rate = record["reference_increment"].sum(-1).mean(-1) / record["interval"]
    reference_map_rate = record["reference_map"].sum((-3, -2, -1)) * record["dx"]**2
    np.testing.assert_allclose(predicted_map_rate, predicted_rate, rtol=2e-5, atol=1e-7)
    np.testing.assert_allclose(reference_map_rate, reference_rate, rtol=2e-5, atol=1e-7)
    print(record["condition_id"], record["predicted_increment"].shape, "map sum OK")
GROUPS = group_records_by_strength(records, include_control=True)
print("비교할 coupling groups:", list(GROUPS))


## Stationarity와 학습 상태

각 g마다 ell과 동일한 g=0 control을 비교한다. production을 네 구간으로 나눠 field variance, conservative energy, medium/active EP drift를 확인한다. 평평해 보이는 것만으로 stationarity가 증명되지는 않는다.

학습 그림은 full validation loss와 실제 사용한 LR, validation 후 LR을 보여준다. 옅은 training loss는 sampled batch 값이라 요동만으로 수렴을 판정하지 않는다. SD는 descriptive spread이며 confidence interval이 아니다.


In [ ]:
for strength, group in GROUPS.items():
    display(Markdown(f"### g={strength:g}: production quarter diagnostics"))
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    for record in group:
        diagnostics = record["diagnostics"]
        variance = np.asarray(diagnostics["variance"])[:, 1:]
        energy = np.asarray(diagnostics["energy"]).sum(-1)[:, 1:]
        mass = np.asarray(diagnostics["mass"])
        area = (record["fields"].shape[-1] * record["dx"])**2
        medium = record["reference_increment"].sum(-1) / record["interval"]
        active = record["reference_increment"][..., 2] / record["interval"]
        quarter_means = lambda values: np.stack(
            [piece.mean(-1) for piece in np.array_split(values, 4, axis=1)], axis=-1)
        quarter = np.arange(1, 5)
        line, = axes[0].plot(quarter, quarter_means(variance).mean(0), "o-", label=condition_label(record))
        axes[1].plot(quarter, quarter_means(energy / area).mean(0), "o-", color=line.get_color())
        axes[2].plot(quarter, quarter_means(medium).mean(0), "o-", color=line.get_color())
        axes[2].plot(quarter, quarter_means(active).mean(0), "s--", color=line.get_color())
        print(record["condition_id"], "max mass drift =", np.max(np.abs(mass - mass[:, :1])))
    axes[0].set(title="Field variance", ylabel="spatial Var(phi)")
    axes[1].set(title="Conservative energy density", ylabel="F_C / area")
    axes[2].set(title="Medium (solid) / active (dashed)", ylabel="global EP / time")
    axes[0].legend(fontsize=8)
    for ax in axes:
        ax.set_xlabel("production quarter")
        ax.set_xticks(np.arange(1, 5))
        ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()

for strength, group in GROUPS.items():
    display(Markdown(f"### g={strength:g}: full validation / LR"))
    fig = plot_training_history(group)
    plt.show()


## 실제 2D kernel과 전체 성능 요약

부호를 보존한 2D kernel을 g,ell 전체에 대해 본다. \(\widehat K(0)=0\)이라 signed weight 합은 0이고 control kernel은 정확히 0이다.

성능 그림은 global mean EP를 **observed step당** 표시하고 KNEEP/direct-medium mean rate ratio를 함께 보여준다. 이는 두 추정량의 비교 지표다. reference 평균이 0에 가까우면 ratio를 정의하지 않는다.


In [ ]:
fig = plot_kernel_diagnostics(records)
plt.show()
fig = plot_ep_performance(records)
plt.show()


## 시간 window별 EP rate

누적 EP 대신 연속된 WINDOW_STEPS=64 observed intervals의 평균 rate를 비교한다. 각 trajectory 시작에서 window를 다시 나누며 independent trajectory를 이어 붙이지 않는다. 마지막 window가 짧으면 실제 길이로 나눈다.


In [ ]:
for strength, group in GROUPS.items():
    display(Markdown(f"### g={strength:g}: window-averaged EP"))
    fig = plot_window_averaged_ep(group, window_steps=WINDOW_STEPS)
    plt.show()


## Time-averaged EP maps

동일 번호 test trajectory의 모든 production interval을 평균한다. linear와 signed-logarithmic(symlog) 색상을 둘 다 표시한다. **음수를 절댓값으로 바꾸지 않는다.** local contribution의 gauge가 달라 pixel별 일치를 목표로 삼지 않는다. map의 공간 합은 해당 global mean rate다.


In [ ]:
for strength, group in GROUPS.items():
    display(Markdown(f"### g={strength:g}: time-averaged maps"))
    for log_scale in (False, True):
        fig = plot_time_averaged_ep(group, trajectory=TEST_TRAJECTORY, log_scale=log_scale)
        plt.show()


## Interaction range별 fluctuation 겹쳐 보기

각 coupling마다 KNEEP끼리, direct medium끼리 ell을 같은 축에 겹친다. **2×2 figure**의 위 행은 KNEEP, 아래 행은 direct medium이며 열은 다음 둘이다.

- \(\operatorname{Var}(j_\rho)\)
- \(\operatorname{Var}(j_\rho)/\rho^2\)

\(j_\rho=\Delta S_\rho/\Delta t_{\rm obs}\)이며 rho=0은 제외한다. 각 trajectory 안에서 temporal variance를 계산한 뒤 평균한다. KNEEP band는 trajectory 평균 후 fit-seed SD, medium band는 독립 test-trajectory SD다. cumulative shell sum은 표시하지 않는다.

아래 shell spectrum 두 열은 signed mean과 temporal variance다. conservative shell의 장시간 평균은 free-energy boundary라 0에 접근할 수 있다. 이것이 긴 interaction range를 부정하지는 않는다.


In [ ]:
for strength, group in GROUPS.items():
    display(Markdown(f"### g={strength:g}: overlaid shell fluctuations"))
    fig = plot_fluctuation_comparison(group)
    plt.show()
    fig = plot_ep_spectrum(group)
    plt.show()


## 선택한 test trajectory: profile / medium EP / predicted EP 영상

v3는 각 조건 첫 test trajectory를 16 observed intervals씩 저장한다. profile은 window 종료 시점 field, medium/predicted EP는 같은 window의 평균 rate다. MOVIE_CONDITION에 출력된 condition ID를 지정하고 MOVIE_KIND를 바꿀 수 있다. combined는 세 영상을 한 화면에서 비교한다. **저장된 평가 수치만** 사용한다.

movie_trajectory는 평가 때 저장한 trajectory 번호다. 다른 번호를 보려면 생성 때 movie_trajectories를 늘리고 평가 때 --movie-trajectory를 지정한다. 기존 결과에 movie가 없으면 설명만 표시한다. CLI movies stage는 profile/medium/KNEEP 각각 MP4(FFmpeg가 없으면 GIF)를 저장한다.


In [ ]:
movie_records = [record for record in records if "movie_fields" in record]
if not movie_records:
    display(Markdown(
        "이 결과에는 window별 medium EP movie가 없습니다. v3에서 movie를 포함해 새로 생성·평가하면 "
        "표시할 수 있습니다. 기존 numerical figure에는 영향이 없습니다."
    ))
else:
    if MOVIE_CONDITION is None:
        selected_movie = next((record for record in movie_records if not record.get("is_control")), movie_records[0])
    else:
        matches = [record for record in movie_records if record["condition_id"] == MOVIE_CONDITION]
        if not matches:
            raise ValueError(f"movie가 있는 condition ID를 선택하세요: {[r['condition_id'] for r in movie_records]}")
        selected_movie = matches[0]
    display(Markdown(
        f"**{condition_label(selected_movie)}**, cached test trajectory "
        f"**{selected_movie.get('movie_trajectory', 0)}**"
    ))
    animation = animate_test_trajectory(
        selected_movie, movie=MOVIE_KIND, log_scale=LOG_SCALE,
        max_frames=MOVIE_MAX_FRAMES, interval_ms=1000 / MOVIE_FPS,
    )
    plt.close(animation._fig)
    display(HTML(animation.to_jshtml()))


## 저장과 해석

CLI all/plot은 figures/에 전체 kernel·performance 및 g별 그림을 저장한다. movie는 자동 MP4/GIF이며 --no-movies로 생략한다. notebook에서는 다음처럼 저장할 수 있다.

    fig = plot_fluctuation_comparison(GROUPS[1.0])
    fig.savefig(OUTPUT_DIR / "fluctuation_g1.png", dpi=200, bbox_inches="tight")
    plt.show()

정상상태 평균 system entropy boundary는 0이다. 평균 EP의 지속적인 차이는 이 항만으로 설명되지 않으며 force 함수족, 유한 데이터, observation/solver 시간 간격, stationarity를 함께 확인한다. 1024-step production으로 저주파 scaling이 검증됐다고 주장하지 않는다. 더 긴 trajectory·독립 seed·더 큰 격자 비교는 별도 실험이다.
